In [1]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
import mlflow
import mlflow.sklearn

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
EXPERIMENT_NAME = "zoomcamp-model"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print(
    "Active experiment:", mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
)

/Users/jinchoi/Documents/Python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow tracking URI: sqlite:///mlflow.db
Active experiment: 1


In [3]:
# Load Jan-Nov 2024 green taxi data
YEAR = 2024
TRAIN_MONTHS = list(range(1, 11))
TEST_MONTH = 11

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year}-{month:02d}.parquet"


def load_month(year: int, month: int) -> pd.DataFrame:
    url = BASE_URL.format(year=year, month=month)
    print(f"  Downloading {year}-{month:02d} ...")
    return pd.read_parquet(url)


def load_months(year: int, months: list) -> pd.DataFrame:
    dfs = [load_month(year, month) for month in months]
    df = pd.concat(dfs, ignore_index=True)
    print(f"  Total: {len(df):,} records\n")
    return df

In [4]:
df_train_raw = load_months(YEAR, TRAIN_MONTHS)
df_test_raw = load_month(YEAR, TEST_MONTH)

print(f"Train raw shape: {df_train_raw.shape}")
print(f"Test raw shape: {df_test_raw.shape}")

  Total: 554,002 records

Train raw shape: (554002, 20)
Test raw shape: (52222, 20)


In [5]:
# Feature setup + split once
FEATURE_COLS = [
    "VendorID",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "trip_type",
    "store_and_fwd_flag",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "is_weekend",
    "rush_hour",
    "trip_duration_min",
    "route",
]

TARGET = "fare_amount"

OHE_COLS = ["VendorID", "RatecodeID", "trip_type", "store_and_fwd_flag"]
TE_COLS = ["route"]
NUM_COLS = [
    "passenger_count",
    "trip_distance",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "is_weekend",
    "rush_hour",
    "trip_duration_min",
]

In [6]:
def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Basic cleaning and feature engineering
    target = "fare_amount"

    df["lpep_pickup_datetime"] = pd.to_datetime(df["lpep_pickup_datetime"])
    df["lpep_dropoff_datetime"] = pd.to_datetime(df["lpep_dropoff_datetime"])

    df["trip_duration_min"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df["pickup_hour"] = df["lpep_pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["lpep_pickup_datetime"].dt.dayofweek
    df["pickup_month"] = df["lpep_pickup_datetime"].dt.month
    df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype(int)
    df["rush_hour"] = df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

    # High-cardinality route feature
    df["route"] = (
        df["PULocationID"].astype("Int64").astype(str)
        + "_"
        + df["DOLocationID"].astype("Int64").astype(str)
    )

    mask = (
        df["trip_duration_min"].between(1, 180)
        & df["fare_amount"].between(0.5, 500)
        & df["trip_distance"].between(0.01, 100)
        & df["passenger_count"].between(0, 8)
    )

    df = df.loc[mask].copy()
    print("Rows after subsetting:", f"{len(df):,}")

    X = df[FEATURE_COLS].copy()
    y = np.log1p(df[target].copy())

    for col in OHE_COLS:
        X[col] = X[col].astype(str)

    return X, y

In [7]:
X_train_full, y_train_full = clean_and_engineer(df_train_raw)
X_test, y_test = clean_and_engineer(df_test_raw)

Rows after subsetting: 495,616
Rows after subsetting: 47,042


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.3, random_state=42
)

print("Train rows:", f"{len(X_train):,}")
print("Val rows  :", f"{len(X_val):,}")
print("Test rows :", f"{len(X_test):,}")

Train rows: 346,931
Val rows  : 148,685
Test rows : 47,042


In [9]:
def compute_vif_statsmodels(df_num: pd.DataFrame) -> pd.DataFrame:
    X = df_num.to_numpy()
    return pd.DataFrame(
        {
            "feature": df_num.columns,
            "vif": [variance_inflation_factor(X, i) for i in range(X.shape[1])],
        }
    ).sort_values("vif", ascending=False)


def vif_prune(df_num: pd.DataFrame, threshold: float = 5.0):
    kept = list(df_num.columns)
    dropped = []

    while len(kept) > 1:
        vif_df = compute_vif_statsmodels(df_num[kept])
        max_row = vif_df.iloc[0]
        max_vif = max_row["vif"]
        max_feature = max_row["feature"]
        if max_vif <= threshold:
            break
        dropped.append({"feature": max_feature, "vif": float(max_vif)})
        kept.remove(max_feature)

    final_vif = compute_vif_statsmodels(df_num[kept])
    return kept, dropped, final_vif

In [10]:
# Train-only numeric imputation + VIF pruning on numeric features
num_imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[NUM_COLS]),
    columns=NUM_COLS,
    index=X_train.index,
)

kept_num_cols, vif_dropped, final_vif = vif_prune(X_train_num, threshold=10.0)
print("Kept numeric columns:", kept_num_cols)
print("Dropped by VIF:", vif_dropped)
print(final_vif)

Kept numeric columns: ['passenger_count', 'trip_distance', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'is_weekend', 'rush_hour', 'trip_duration_min']
Dropped by VIF: []
             feature       vif
3   pickup_dayofweek  6.718823
7  trip_duration_min  6.004362
2        pickup_hour  5.337067
1      trip_distance  4.377975
4       pickup_month  3.845875
5         is_weekend  3.104782
0    passenger_count  2.536565
6          rush_hour  1.737054


In [11]:
# Preprocessor factory with post-VIF numeric columns
def make_preprocessor(kept_num_cols):
    num_pipe = Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    ohe_pipe = Pipeline(
        [
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    te_pipe = Pipeline(
        [
            (
                "te",
                TargetEncoder(
                    cols=TE_COLS,
                    smoothing=20,
                    handle_missing="value",
                    handle_unknown="value",
                ),
            ),
        ]
    )

    return ColumnTransformer(
        [
            ("num", num_pipe, kept_num_cols),
            ("ohe", ohe_pipe, OHE_COLS),
            ("te", te_pipe, TE_COLS),
        ]
    )


def rmse_from_log(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [12]:
# Baseline metric (no VIF/LASSO): Ridge on all numeric cols + OHE + TE
baseline_preprocessor = make_preprocessor(NUM_COLS)
baseline_model = Pipeline(
    [
        ("preprocess", baseline_preprocessor),
        ("reg", Ridge(alpha=1.0, random_state=42)),
    ]
)
baseline_model.fit(X_train, y_train)

baseline_val_pred_log = baseline_model.predict(X_val)
baseline_val_rmse = rmse_from_log(y_val, baseline_val_pred_log)
print(f"Baseline val RMSE (no VIF/LASSO): {baseline_val_rmse:.4f}")

Baseline val RMSE (no VIF/LASSO): 45.4951


In [13]:
# LASSO selection on post-VIF feature space (train only), tune alpha on validation
preprocessor_vif = make_preprocessor(kept_num_cols)
alphas = np.logspace(-4, 0, 12)

best_alpha = None
best_lasso_model = None
best_lasso_val_rmse = float("inf")
min_selected_features = 2

fallback_best_alpha = None
fallback_best_model = None
fallback_best_rmse = float("inf")

with mlflow.start_run(run_name="lasso_hyperparameter_tuning") as lasso_parent_run:
    mlflow.log_param("stage", "lasso_hpo")
    mlflow.log_param("min_selected_features", min_selected_features)
    mlflow.log_param("alpha_grid", ",".join([f"{a:.8f}" for a in alphas]))

    for alpha in alphas:
        with mlflow.start_run(run_name=f"lasso_alpha_{alpha:.6f}", nested=True):
            lasso_model = Pipeline(
                [
                    ("preprocess", preprocessor_vif),
                    ("reg", Lasso(alpha=float(alpha), max_iter=20000, random_state=42)),
                ]
            )
            lasso_model.fit(X_train, y_train)

            val_pred_log = lasso_model.predict(X_val)
            val_rmse = rmse_from_log(y_val, val_pred_log)
            n_selected = int(
                (np.abs(lasso_model.named_steps["reg"].coef_) > 1e-8).sum()
            )

            mlflow.log_param("alpha", float(alpha))
            mlflow.log_metric("val_rmse", val_rmse)
            mlflow.log_metric("n_selected_features", n_selected)

            print(f"alpha={alpha:.6f}  val_rmse={val_rmse:.4f}  selected={n_selected}")

            if val_rmse < fallback_best_rmse:
                fallback_best_rmse = val_rmse
                fallback_best_alpha = float(alpha)
                fallback_best_model = lasso_model

            if n_selected >= min_selected_features and val_rmse < best_lasso_val_rmse:
                best_lasso_val_rmse = val_rmse
                best_alpha = float(alpha)
                best_lasso_model = lasso_model

    if best_lasso_model is None:
        print(
            f"No alpha met min_selected_features={min_selected_features}; using RMSE-best fallback."
        )
        best_alpha = fallback_best_alpha
        best_lasso_model = fallback_best_model
        best_lasso_val_rmse = fallback_best_rmse

    mlflow.log_metric("best_lasso_val_rmse", best_lasso_val_rmse)
    mlflow.log_param("best_lasso_alpha", best_alpha)

print(f"Best LASSO alpha: {best_alpha:.6f}")
print(f"Post-VIF + LASSO val RMSE: {best_lasso_val_rmse:.4f}")

alpha=0.000100  val_rmse=43.8406  selected=14
alpha=0.000231  val_rmse=43.0929  selected=12
alpha=0.000534  val_rmse=42.2102  selected=11
alpha=0.001233  val_rmse=41.7305  selected=10
alpha=0.002848  val_rmse=45.5084  selected=8
alpha=0.006579  val_rmse=56.3320  selected=5
alpha=0.015199  val_rmse=86.5108  selected=3
alpha=0.035112  val_rmse=207.1280  selected=3
alpha=0.081113  val_rmse=302.4246  selected=2
alpha=0.187382  val_rmse=27.1355  selected=2
alpha=0.432876  val_rmse=14.5057  selected=2
alpha=1.000000  val_rmse=14.8832  selected=0
Best LASSO alpha: 0.432876
Post-VIF + LASSO val RMSE: 14.5057


In [16]:
best_lasso_model[:-1]

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse

In [17]:
best_lasso_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [ ]:
# Extract selected features from best LASSO and run consistency checks
feature_names = best_lasso_model.named_steps["preprocess"].get_feature_names_out()
coefs = best_lasso_model.named_steps["reg"].coef_
coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})

selected_mask = np.abs(coefs) > 1e-8
selected_feature_names = feature_names[selected_mask].tolist()

# Verify transform compatibility on val/test and stable feature order
X_train_tx = best_lasso_model[:-1].transform(X_train)
X_val_tx = best_lasso_model[:-1].transform(X_val)

coef_df.loc[selected_mask, "abs_coef"] = coef_df["coef"].abs()

print("Total transformed features:", len(feature_names))
print("Selected non-zero features:", len(selected_feature_names))
print("Top selected features:")
print(coef_df.loc[selected_mask])

Total transformed features: 23
Selected non-zero features: 2
Top selected features:
                  feature      coef  abs_coef
1      num__trip_distance  0.023318  0.023318
7  num__trip_duration_min  0.002959  0.002959


In [ ]:
# Final model: tune Ridge on selected features, then retrain on train+val, evaluate once on test
selected_indices = [
    int(np.where(feature_names == name)[0][0]) for name in selected_feature_names
]

X_train_sel = X_train_tx[:, selected_indices]
X_val_sel = X_val_tx[:, selected_indices]

ridge_alphas = np.logspace(-4, 4, 50)
best_ridge_alpha = None
best_ridge_val_rmse = float("inf")

with mlflow.start_run(run_name="ridge_training_and_final_model") as ridge_run:
    mlflow.log_param("stage", "ridge_hpo_final")
    mlflow.log_param("selected_feature_count", len(selected_feature_names))
    mlflow.log_param("ridge_alpha_grid_size", len(ridge_alphas))

    for alpha in ridge_alphas:
        with mlflow.start_run(run_name=f"ridge_alpha_{alpha:.6f}", nested=True):
            ridge = Ridge(alpha=alpha, random_state=42)
            ridge.fit(X_train_sel, y_train)
            val_pred_log = ridge.predict(X_val_sel)
            val_rmse = rmse_from_log(y_val, val_pred_log)

            mlflow.log_param("alpha", float(alpha))
            mlflow.log_metric("val_rmse", val_rmse)

            print(f"Ridge alpha={alpha:.2f}  val_rmse={val_rmse:.4f}")
            if val_rmse < best_ridge_val_rmse:
                best_ridge_val_rmse = val_rmse
                best_ridge_alpha = alpha

    print(f"Best Ridge alpha (selected vars): {best_ridge_alpha}")
    print(f"Final Ridge-on-selected val RMSE: {best_ridge_val_rmse:.4f}")

    # Refit once on train+val

    final_preprocessor = make_preprocessor(kept_num_cols)
    X_trainval_tx = final_preprocessor.fit_transform(X_trainval, y_trainval)
    final_feature_names = final_preprocessor.get_feature_names_out()

    name_to_idx = {name: i for i, name in enumerate(final_feature_names)}
    final_selected_indices = [name_to_idx[name] for name in selected_feature_names]

    X_trainval_sel = X_trainval_tx[:, final_selected_indices]
    final_ridge = Ridge(alpha=best_ridge_alpha, random_state=42)
    final_ridge.fit(X_trainval_sel, y_trainval)

    # Evaluate once on test
    X_test_tx_final = final_preprocessor.transform(X_test)
    X_test_sel_final = X_test_tx_final[:, final_selected_indices]
    test_pred_log = final_ridge.predict(X_test_sel_final)
    final_test_rmse = rmse_from_log(y_test, test_pred_log)
    print(f"Final test RMSE (Ridge on selected vars): {final_test_rmse:.4f}")

    mlflow.log_param("best_ridge_alpha", float(best_ridge_alpha))
    mlflow.log_metric("best_ridge_val_rmse", best_ridge_val_rmse)
    mlflow.log_metric("final_test_rmse", final_test_rmse)
    mlflow.log_param("selected_feature_count", len(selected_feature_names))

    mlflow.log_dict(
        {"selected_feature_names": selected_feature_names}, "selected_features.json"
    )
    mlflow.sklearn.log_model(final_ridge, artifact_path="final_model")
    logged_model_uri = f"runs:/{ridge_run.info.run_id}/final_model"

print("Logged model URI:", logged_model_uri)

Ridge alpha=0.00  val_rmse=1296.4794
Ridge alpha=0.00  val_rmse=1296.4794
Ridge alpha=0.00  val_rmse=1296.4794
Ridge alpha=0.00  val_rmse=1296.4794
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.00  val_rmse=1296.4793
Ridge alpha=0.01  val_rmse=1296.4792
Ridge alpha=0.01  val_rmse=1296.4791
Ridge alpha=0.01  val_rmse=1296.4790
Ridge alpha=0.02  val_rmse=1296.4789
Ridge alpha=0.03  val_rmse=1296.4787
Ridge alpha=0.04  val_rmse=1296.4783
Ridge alpha=0.06  val_rmse=1296.4779
Ridge alpha=0.09  val_rmse=1296.4772
Ridge alpha=0.13  val_rmse=1296.4762
Ridge alpha=0.18  val_rmse=1296.4748
Ridge alpha=0.27  val_rmse=1296.4727
Ridge alpha=0.39  val_rmse=1296.4697
Ridge alpha=0.57  val_rmse=1296.4653
Ridge alpha=0.83  val_rmse=1296.4589
Ridge alpha=1.21  val_rmse=1296.4495
Ridge alpha=1.76  val_rmse=1296.4359
R

2026/03/08 20:30:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final test RMSE (Ridge on selected vars): 416.7721
Logged model URI: runs:/4792df35c83c403ebd7fe138b8c941b8/final_model


In [ ]:
# Inference-time: load model from MLflow and evaluate
loaded_model = mlflow.sklearn.load_model(logged_model_uri)
X_test_tx_loaded = final_preprocessor.transform(X_test)
X_test_sel_loaded = X_test_tx_loaded[:, final_selected_indices]
loaded_test_pred_log = loaded_model.predict(X_test_sel_loaded)
loaded_test_rmse = rmse_from_log(y_test, loaded_test_pred_log)

print("Loaded model URI:", logged_model_uri)
print(f"Loaded-model test RMSE: {loaded_test_rmse:.4f}")

Loaded model URI: runs:/4792df35c83c403ebd7fe138b8c941b8/final_model
Loaded-model test RMSE: 416.7721


In [ ]:
# Optional: register the latest logged model
registered_model_name = "nyc-taxi-ridge"
model_details = mlflow.register_model(
    model_uri=logged_model_uri, name=registered_model_name
)
print("Registered model version:", model_details.version)

Registered model 'nyc-taxi-ridge' already exists. Creating a new version of this model...
2026/03/08 20:33:36 WARNING mlflow.tracking._model_registry.fluent: Run with id 4792df35c83c403ebd7fe138b8c941b8 has no artifacts at artifact path 'final_model', registering model based on models:/m-6d95f0382ae84aaba1d0ce2149988f3e instead


Registered model version: 4


Created version '4' of model 'nyc-taxi-ridge'.
